# Battery Management System (BMS) Optimization & Safety Fault Detection
### Comprehensive Presentation & Demonstration Notebook

**Live Demonstration Structure:**
1. **Part 1: EV Fast-Charging Thermal Runaway Telemetry (1,000 samples, 17 sensors)**
   - Data Ingestion & Exploratory Analytics
   - Transparent Preprocessing, Encoding & Stratified Splitting (Code written out in cell)
   - Live Swarm Executions: BPSO, BWOA, BHPWOA, and Proposed HGWO-PSO
   - Inline Visualizations: Convergence Curves & SHAP Explainability
   - Live Performance Comparison Table
2. **Part 2: NASA Li-ion Classic 4-Cell Benchmark (636 cycles, 24°C)**
   - Transparent Preprocessing & Live Swarm Optimizations
   - Benchmark Comparison Table
3. **Part 3: NASA Global Multi-Temperature Archive (2,750 cycles, 34 cells, 4°C - 44°C)**
   - Extreme Environmental Temperature Robustness Testing
   - Benchmark Comparison Table
4. **Part 4: Multi-Dataset Cross-Validation Plots**

In [ ]:
# Step 1: Clone Repository & Setup Environment
import os
if not os.path.exists("BMS"):
    !git clone https://github.com/abdullahgni/BMS.git
    %cd BMS
else:
    %cd BMS
    !git pull origin main

!pip install -q -r requirements.txt

---
## PART 1: EV Fast-Charging Thermal Runaway Telemetry

In [ ]:
# Step 2: Telemetry Data Ingestion & Analytics
import pandas as pd
import numpy as np

df_ev = pd.read_csv("EV_Battery_Charging_TR_Dataset_with_Notes.csv")
print("Dataset Shape:", df_ev.shape)
print("
--- First 5 Records ---")
display(df_ev.head())
print("
--- Telemetry Schema & Data Types ---")
df_ev.info()
print("
--- Statistical Distribution Summary ---")
display(df_ev.describe())

In [ ]:
# Step 3: Transparent Preprocessing, Encoding & Stratified Split Setup
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold

# 1. Encode discrete protocol features
le = LabelEncoder()
df_ev["ChargingStage"] = le.fit_transform(df_ev["ChargingStage"])
df_ev["MoistureDetected"] = df_ev["MoistureDetected"].astype(int)

# 2. Map ground-truth physical safety targets:
# 0: Critical (Runaway), 1: OK (Watch), 2: Warning (Alarm/Warning)
target_map = {"Runaway": 0, "Watch": 1, "Alarm": 2, "Warning": 2}
y_ev = df_ev["EventFlag"].map(target_map).values

# 3. Discard non-sensor metadata columns
metadata_cols = ["Timestamp", "ChargerID", "CellID", "Notes", "BMS_Status", "EventFlag", "TR_Probability"]
X_ev = df_ev.drop(columns=metadata_cols)
feature_names_ev = X_ev.columns.tolist()

# 4. Standardize physical sensor channels (Z-Score scaling)
scaler_ev = StandardScaler()
X_ev_scaled = pd.DataFrame(scaler_ev.fit_transform(X_ev), columns=feature_names_ev)

# 5. Initialize Stratified 5-Fold Cross-Validation splitter
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Candidate Physical Sensors ({len(feature_names_ev)}): {feature_names_ev}")
print(f"Target Distribution: Critical (0) = {(y_ev == 0).sum()}, OK (1) = {(y_ev == 1).sum()}, Warning (2) = {(y_ev == 2).sum()}")
display(X_ev_scaled.head())

In [ ]:
# Step 4: Run Live BPSO (Binary Particle Swarm Optimization)
!python bpso.py --data EV_Battery_Charging_TR_Dataset_with_Notes.csv

In [ ]:
# Step 5: Run Live BWOA (Binary Whale Optimization Algorithm)
!python bwoa.py --data EV_Battery_Charging_TR_Dataset_with_Notes.csv

In [ ]:
# Step 6: Run Live BHPWOA (Binary Hybrid PSO-WOA)
!python bhpwoa.py --data EV_Battery_Charging_TR_Dataset_with_Notes.csv

In [ ]:
# Step 7: Run Live Proposed HGWO-PSO (Wolf Pack Co-Optimization)
!python hgwo_pso.py --data EV_Battery_Charging_TR_Dataset_with_Notes.csv

In [ ]:
# Step 8: Multi-Objective Convergence Curves (Direct Inline Display)
from plot_convergence import plot_convergence
plot_convergence()

In [ ]:
# Step 9: SHAP Explainable AI (XAI) Feature Attribution (Direct Inline Display)
from plot_shap import run_shap_analysis
run_shap_analysis("EV_Battery_Charging_TR_Dataset_with_Notes.csv")

In [ ]:
# Step 10: Final Comparison Benchmark Table (EV Dataset)
!python benchmark_comparison.py --data EV_Battery_Charging_TR_Dataset_with_Notes.csv

---
## PART 2: NASA Li-ion Classic 4-Cell Benchmark (636 cycles at 24°C)
Testing cross-dataset generalizability on experimental battery cells (B0005, B0006, B0007, B0018).

In [ ]:
# Step 11: Ingest NASA Classic 4 Dataset & Transparent Preprocessing
df_n4 = pd.read_csv("nasa_battery_benchmark_classic4.csv")
print("NASA Classic 4 Shape:", df_n4.shape)
display(df_n4.head())

# Preprocessing
non_features_n4 = ["battery_id", "cycle_file", "Capacity_Ah", "Health_State", "Target_Label"]
feat_n4 = [c for c in df_n4.columns if c not in non_features_n4]
y_n4 = df_n4["Target_Label"].values

scaler_n4 = StandardScaler()
X_n4_scaled = pd.DataFrame(scaler_n4.fit_transform(df_n4[feat_n4]), columns=feat_n4)
print(f"NASA Candidate Features ({len(feat_n4)}): {feat_n4}")
print(f"Target Distribution: Critical EOL (0) = {(y_n4 == 0).sum()}, OK (1) = {(y_n4 == 1).sum()}, Warning (2) = {(y_n4 == 2).sum()}")
display(X_n4_scaled.head())

In [ ]:
# Step 12: Run Algorithms on NASA Classic 4 (Live Swarms)
!python bpso.py --data nasa_battery_benchmark_classic4.csv
!python bwoa.py --data nasa_battery_benchmark_classic4.csv
!python hgwo_pso.py --data nasa_battery_benchmark_classic4.csv

In [ ]:
# Step 13: NASA Classic 4 Comparison Table
!python benchmark_comparison.py --data nasa_battery_benchmark_classic4.csv

---
## PART 3: NASA Global Multi-Temperature Archive (2,750 cycles, 34 cells, 4°C - 44°C)
Evaluating environmental robustness across sub-freezing winter chambers (4°C), room temperature (24°C), and summer heat stress (44°C).

In [ ]:
# Step 14: Ingest NASA Global Archive & Transparent Preprocessing
df_ng = pd.read_csv("nasa_battery_benchmark_global_2750.csv")
print("NASA Global Archive Shape:", df_ng.shape)
display(df_ng.head())
df_ng.info()

# Preprocessing
non_features_ng = ["battery_id", "cycle_file", "Capacity_Ah", "Health_State", "Target_Label"]
feat_ng = [c for c in df_ng.columns if c not in non_features_ng]
y_ng = df_ng["Target_Label"].values

scaler_ng = StandardScaler()
X_ng_scaled = pd.DataFrame(scaler_ng.fit_transform(df_ng[feat_ng]), columns=feat_ng)
print(f"Global Features ({len(feat_ng)}): {feat_ng}")
print(f"Target Distribution across 4°C-44°C: Critical EOL (0) = {(y_ng == 0).sum()}, OK (1) = {(y_ng == 1).sum()}, Warning (2) = {(y_ng == 2).sum()}")
display(X_ng_scaled.head())

In [ ]:
# Step 15: Run Algorithms on NASA Global Multi-Temperature Archive (Live Swarms)
!python bpso.py --data nasa_battery_benchmark_global_2750.csv
!python bwoa.py --data nasa_battery_benchmark_global_2750.csv
!python hgwo_pso.py --data nasa_battery_benchmark_global_2750.csv

In [ ]:
# Step 16: NASA Global Multi-Temperature Comparison Table
!python benchmark_comparison.py --data nasa_battery_benchmark_global_2750.csv

---
## PART 4: Cross-Dataset Multi-Panel Performance Summary

In [ ]:
# Step 17: Cross-Dataset Multi-Panel Plots (Direct Inline Display)
from plot_cross_dataset import plot_cross_dataset
plot_cross_dataset()